# VisionGym Baseline
Colab GPU에서 synthetic benchmark를 생성하고 Qwen3-VL-2B-Instruct를 zero-shot / prompt variant로 평가합니다.

In [ ]:
# 공개 저장소를 Colab에 가져오고 VLM inference 의존성을 설치합니다.
!git clone -q https://github.com/oosuhada/visiongym.git /content/visiongym || true
%cd /content/visiongym
!git pull -q
!pip install -q -e '.[vlm]'

In [ ]:
# 데이터 생성은 CPU 작업이므로 GPU 시간을 쓰기 전에 완료합니다.
!visiongym generate --config configs/dataset.yaml --output data/generated
!python - <<'PY'
import json
from pathlib import Path
manifest = json.loads(Path('data/generated/manifest.json').read_text())
print('benchmark QA:', manifest['benchmark_qa_pairs'])
for split in manifest['splits']:
    print(split['split'], split['scenes'], split['qa_pairs'])
PY

In [ ]:
# 기본 direct-answer prompt로 전체 ID + OOD benchmark를 평가합니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/base-direct.jsonl --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode direct --load-in-4bit
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/base-direct.jsonl --output reports/base-direct --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode direct
!visiongym report --metrics reports/base-direct/metrics.json --output reports/base-direct

In [ ]:
# Prompt만 바꿨을 때의 효과를 같은 benchmark에서 비교합니다.
import subprocess
for mode in ['json', 'reasoning', 'fewshot']:
    subprocess.run(['visiongym', 'infer', '--dataset', 'data/generated/benchmark.jsonl', '--output', f'outputs/base-{mode}.jsonl', '--model', 'Qwen/Qwen3-VL-2B-Instruct', '--prompt-mode', mode, '--load-in-4bit'], check=True)
    subprocess.run(['visiongym', 'evaluate', '--dataset', 'data/generated/benchmark.jsonl', '--predictions', f'outputs/base-{mode}.jsonl', '--output', f'reports/base-{mode}', '--model', 'Qwen/Qwen3-VL-2B-Instruct', '--prompt-mode', mode], check=True)
!visiongym compare reports/base-direct/metrics.json reports/base-json/metrics.json reports/base-reasoning/metrics.json reports/base-fewshot/metrics.json --output reports/prompt-comparison.csv